# Tree Models
### The following models will be tested
- Random Forest
- XGBoost

In [17]:
import pandas as pd
import numpy as np
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import gc

### Load and Clean Data

In [10]:
print("--- Loading Data ---")

# A. Load Sample Data
raw_train_sample_full = pd.read_csv('train_sample_1.csv')
raw_train_sample = raw_train_sample_full.sample(frac = 0.5, random_state=42)

# B. Load Full Data & Test Data
raw_train_full = pd.read_csv('train_1.csv')
raw_test = pd.read_csv('test_1.csv')

--- Loading Data ---


In [18]:
# Clean data
cols_to_drop = [
    'trips_ended', 'net_flow', 
    'station_id', 'station_name', 
    'date', 'datetime'
]

# Drop unnecessary columns
clean_train_sample = raw_train_sample.drop(columns=cols_to_drop, errors='ignore')
clean_train_full = raw_train_full.drop(columns=cols_to_drop, errors='ignore')
clean_test = raw_test.drop(columns=cols_to_drop, errors='ignore')

print(f"Sample Train Shape: {clean_train_sample.shape}")
print(f"Full Train Shape:   {clean_train_full.shape}")
print(f"Test Shape:   {clean_test.shape}")

Sample Train Shape: (640152, 42)
Full Train Shape:   (12803054, 42)
Test Shape:   (3174974, 42)


### Preprocess Data

In [19]:
# Separate Target (y) and Features (X)
# Sample
X_train_sample = clean_train_sample.drop(columns='trips_started')
y_train_sample = clean_train_sample['trips_started']

# Full
X_train_full = clean_train_full.drop(columns='trips_started')
y_train_full = clean_train_full['trips_started']

# Test
X_test = clean_test.drop(columns='trips_started')
y_test = clean_test['trips_started']

print(f"Features used for RF: {X_train_sample.shape[1]}")

Features used for RF: 41


## Random Forest

### Hyperparameter Tuning

In [25]:
print("\n--- Tuning Random Forest on Sample Data ---")

# Define the grid
param_dist = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15, 18],
    'min_samples_split': [20, 50, 100],
    'min_samples_leaf': [10, 20, 50],
    'max_features': ['sqrt', 0.3]
}

rf_tuner = RandomForestRegressor(
    random_state=42, 
    n_jobs=1, # Changed to use 1 core to limit RAM usage
    max_samples=0.8
) 

# Run Random Search
random_search = RandomizedSearchCV(
    estimator=rf_tuner,
    param_distributions=param_dist,
    n_iter=10, 
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    random_state=42,
    n_jobs=6 # Changed to use 6 cores to limit RAM usage
)

random_search.fit(X_train_sample, y_train_sample)
best_params_rf = random_search.best_params_
print(f"Best Parameters: {best_params_rf}")


--- Tuning Random Forest on Sample Data ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits


python(51925) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51926) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51927) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51928) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51929) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51930) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Best Parameters: {'n_estimators': 100, 'min_samples_split': 100, 'min_samples_leaf': 10, 'max_features': 0.3, 'max_depth': 18}


### Training and Evaluation

In [26]:
# --- Model A: Trained on Sample Data ---
print("\n--- Training RF on Sample Data ---")
rf_model_sample = RandomForestRegressor(
    **best_params_rf, 
    random_state=42, 
    n_jobs=4,
    verbose=2
)
rf_model_sample.fit(X_train_sample, y_train_sample)

# Predict on Test
y_pred_sample_on_test = rf_model_sample.predict(X_test)

# --- Model B: Trained on Full Data ---
print("\n--- Training RF on Full Data ---")
rf_model_full = RandomForestRegressor(
    **best_params_rf, 
    random_state=42, 
    n_jobs=4,
    verbose=2
)
rf_model_full.fit(X_train_full, y_train_full)


--- Training RF on Sample Data ---
building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100building tree 38 of 100

building tree 39 of 100
building tree 40 of 100


[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   13.6s


building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82

...

building tree 1 of 100building tree 2 of 100

building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100


[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:  6.7min


building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79

[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed: 19.3min finished


,n_estimators,100
,criterion,'squared_error'
,max_depth,18
,min_samples_split,100
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,0.3
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### Generate Predictions

In [27]:
# --- Sample Model Predictions ---
# Train set prediction
y_pred_sample_train = rf_model_sample.predict(X_train_sample)
# Test set prediction
y_pred_sample_on_test = rf_model_sample.predict(X_test)

# --- Full Model Predictions ---
# Train set prediction (This might take a moment on 15M rows)
y_pred_full_train = rf_model_full.predict(X_train_full)
# Test set prediction
y_pred_full_on_test = rf_model_full.predict(X_test)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    1.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    3.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    6.7s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:   19.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    1.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.0s finished


### Calculate Metrics and Evaluate

In [28]:
# Sample Model Scores
rmse_sample_train = np.sqrt(mean_squared_error(y_train_sample, y_pred_sample_train))
r2_sample_train = r2_score(y_train_sample, y_pred_sample_train)

rmse_sample_test = np.sqrt(mean_squared_error(y_test, y_pred_sample_on_test))
r2_sample_test = r2_score(y_test, y_pred_sample_on_test)

# Full Model Scores
rmse_full_train = np.sqrt(mean_squared_error(y_train_full, y_pred_full_train))
r2_full_train = r2_score(y_train_full, y_pred_full_train)

rmse_full_test = np.sqrt(mean_squared_error(y_test, y_pred_full_on_test))
r2_full_test = r2_score(y_test, y_pred_full_on_test)

print("\n" + "="*60)
print("MODEL CONFIGURATION (Best Params)")
print("="*60)
for param, value in best_params_rf.items():
    print(f"{param:<25}: {value}")

print("\n" + "="*60)
print(f"{'METRIC':<10} | {'RF (Sample Trained)':<20} | {'RF (Full Trained)':<20}")
print("="*60)
print(f"{'Train RMSE':<10} | {rmse_sample_train:<20.4f} | {rmse_full_train:<20.4f}")
print(f"{'Test RMSE':<10} | {rmse_sample_test:<20.4f} | {rmse_full_test:<20.4f}")
print("-" * 60)
print(f"{'Train R^2':<10} | {r2_sample_train:<20.4f} | {r2_full_train:<20.4f}")
print(f"{'Test R^2':<10} | {r2_sample_test:<20.4f} | {r2_full_test:<20.4f}")
print("="*60)


MODEL CONFIGURATION (Best Params)
n_estimators             : 100
min_samples_split        : 100
min_samples_leaf         : 10
max_features             : 0.3
max_depth                : 18

METRIC     | RF (Sample Trained)  | RF (Full Trained)   
Train RMSE | 1.2209               | 1.1150              
Test RMSE  | 1.2777               | 1.1432              
------------------------------------------------------------
Train R^2  | 0.5681               | 0.6376              
Test R^2   | 0.5356               | 0.6282              


## XGBoost

### Device Setup

In [ ]:
# 1. Detect Device
# We use PyTorch to check for Nvidia GPU (CUDA) availability
if torch.cuda.is_available():
    print(f"NVIDIA GPU Detected: {torch.cuda.get_device_name(0)}")
    target_device = "cuda"
    n_jobs_xgb = 1  # GPU handles the parallelism internally, so we set CPU jobs to 1
else:
    print("No NVIDIA GPU found. Using CPU.")
    target_device = "cpu"
    n_jobs_xgb = 6  # Set to 4 or 6 based on your 25GB RAM

print(f"Models will run on: {target_device.upper()}")

### Hyperparameter Tuning

In [ ]:
print("\n--- Tuning XGBoost on Sample Data ---")

# XGBoost specific hyperparameters
param_dist = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],   # Lower rate + more trees usually wins
    'max_depth': [6, 10, 15],             # XGBoost prefers shallower trees than RF
    'subsample': [0.6, 0.8, 1.0],         # Row sampling to prevent overfitting
    'colsample_bytree': [0.6, 0.8, 1.0],   # Feature sampling (like max_features in RF)
    'gamma': [0, 0.5, 1],                   # minimum loss reduction to make a split
    'reg_lambda': [1, 5, 10]                # L2 regularization strength
}

xgb_tuner = XGBRegressor(
    target_device=target_device,
    random_state=42, 
    n_jobs=n_jobs_xgb,
    objective='reg:squarederror' # Explicitly set objective for regression
)

random_search = RandomizedSearchCV(
    estimator=xgb_tuner,
    param_distributions=param_dist,
    n_iter=15,           # XGB is faster than RF, so we can try a few more combos
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_sample, y_train_sample)
best_params_xgb = random_search.best_params_
print(f"Best Parameters: {best_params_xgb}")

### Training and Evaluation

In [ ]:
# --- Model A: Trained on Sample Data ---
print("\n--- Training XGBoost on Sample Data ---")
xgb_model_sample = XGBRegressor(
    **best_params_xgb,
    random_state=42,
    n_jobs=-1, 
    objective='reg:squarederror'
)
xgb_model_sample.fit(X_train_sample, y_train_sample)

# Predict on Test
y_pred_sample_on_test = xgb_model_sample.predict(X_test)

# --- Model B: Trained on Full Data ---
print("\n--- Training XGBoost on Full Data ---")
xgb_model_full = XGBRegressor(
    **best_params_xgb, 
    random_state=42, 
    n_jobs=-1, 
    objective='reg:squarederror'
)
xgb_model_full.fit(X_train_full, y_train_full)

### Generate Predictions

In [ ]:
# 1. Full Model Predictions
y_pred_full_on_train = xgb_model_full.predict(X_train_full) # NEW: Predict on Train
y_pred_full_on_test  = xgb_model_full.predict(X_test)

# 2. Sample Model Predictions 
y_pred_sample_on_train = xgb_model_sample.predict(X_train_sample)
y_pred_sample_on_test  = xgb_model_sample.predict(X_test)

### Metrics and Evaluation

In [ ]:
# Sample Model Scores
rmse_sample_train_xgb = np.sqrt(mean_squared_error(y_train_sample, y_pred_sample_on_train))
r2_sample_train_xgb   = r2_score(y_train_sample, y_pred_sample_on_train)

rmse_sample_test_xgb  = np.sqrt(mean_squared_error(y_test, y_pred_sample_on_test))
r2_sample_test_xgb    = r2_score(y_test, y_pred_sample_on_test)

# Full Model Scores
rmse_full_train_xgb   = np.sqrt(mean_squared_error(y_train_full, y_pred_full_on_train))
r2_full_train_xgb     = r2_score(y_train_full, y_pred_full_on_train)

rmse_full_test_xgb    = np.sqrt(mean_squared_error(y_test, y_pred_full_on_test))
r2_full_test_xgb      = r2_score(y_test, y_pred_full_on_test)

# Final Report
print("\n" + "="*60)
print(f"{'METRIC':<10} | {'XGB (Sample Trained)':<20} | {'XGB (Full Trained)':<20}")
print("="*60)
print(f"{'Train RMSE':<10} | {rmse_sample_train_xgb:<20.4f} | {rmse_full_train_xgb:<20.4f}")
print(f"{'Test RMSE':<10} | {rmse_sample_test_xgb:<20.4f} | {rmse_full_test_xgb:<20.4f}")
print("-" * 60)
print(f"{'Train R^2':<10} | {r2_sample_train_xgb:<20.4f} | {r2_full_train_xgb:<20.4f}")
print(f"{'Test R^2':<10} | {r2_sample_test_xgb:<20.4f} | {r2_full_test_xgb:<20.4f}")
print("="*60)

In [ ]:
# Clean up RAM
del raw_train_full, clean_train_full, X_train_full, y_train_full
gc.collect()